# Advanced SQL Examples and Exercises: ANSWERS

These are the answers to the 💻 tasks in `Advanced_SQL_e-learning_examples_and_exercises.ipynb`, translated from T-SQL to SQLite in the same way as the examples notebook. Look out for the **SQLite note** boxes where the SQL differs from the original T-SQL answers.

Some answers build on tables, functions or other objects that the examples notebook creates. So that this notebook runs on its own, those set-up steps are repeated here, in cells marked *Set-up from the examples notebook*.

## Setup

Run these cells first. They install `jupysql` and load a fresh copy of the Sakila database into memory.

In [ ]:
# Hide some harmless but distracting warnings from jupysql
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
import sqlite3
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

# Copy sakila.sqlite into an in-memory database, so every session starts with the original data
source = sqlite3.connect('sakila.sqlite')
con = sqlite3.connect(':memory:', check_same_thread=False)
source.backup(con)
source.close()

# Connect jupysql to that one connection
engine = create_engine('sqlite://', creator=lambda: con, poolclass=StaticPool)
%sql engine

---

## 01 INSERT INTO

💻 **Task:** Modify the existing INSERT statement to add two new categories instead of one: Mystery and Romance. Ensure that both new categories are added in a single INSERT INTO statement. After modifying the INSERT statement, update the subsequent SELECT FROM WHERE statements to verify both insertions. Finally, modify the DELETE statement to delete both newly added categories.

📗 How to insert multiple rows of data https://learn.microsoft.com/en-us/sql/t-sql/statements/insert-transact-sql?view=sql-server-ver16#b-inserting-multiple-rows-of-data

In [ ]:
%%sql
INSERT INTO category (name)
VALUES ('Mystery'), ('Romance');

Confirm the insertion by counting rows again.

In [ ]:
%%sql
SELECT COUNT(*) as n
FROM category;

View the rows for the inserted categories, using 'IN' to filter for both categories in a single query.

In [ ]:
%%sql
SELECT *
FROM category
WHERE name IN ('Mystery', 'Romance');

Delete the newly added rows for cleanup, again using 'IN' to delete both categories in one statement.

In [ ]:
%%sql
DELETE FROM category
WHERE name IN ('Mystery', 'Romance');

Confirm the deletion by counting rows again.

In [ ]:
%%sql
SELECT COUNT(*) as n
FROM category;

---

## 02 INSERT INTO with SELECT

💻 **Task:** Modify the query to insert actors into 'TopActors' who have appeared in more than 10 films, instead of more than 5. Change the HAVING clause in both the SELECT statement used for review and the INSERT INTO statement to reflect this new criteria.

Check if the 'TopActors' table already exists and drop it if necessary, then create the TopActors table.

In [ ]:
%%sql
DROP TABLE IF EXISTS TopActors;

CREATE TABLE TopActors (
    ActorID INT,
    FirstName VARCHAR(45),
    LastName VARCHAR(45),
    TotalFilms INT
);

For review and checking the data before insertion:

In [ ]:
%%sql
SELECT
    a.actor_id,
    a.first_name,
    a.last_name,
    COUNT(fa.film_id) AS TotalFilms
FROM
    actor a
JOIN
    film_actor fa ON a.actor_id = fa.actor_id
GROUP BY
    a.actor_id, a.first_name, a.last_name
HAVING
    COUNT(fa.film_id) > 10; -- Changed from > 5 to > 10

For actual insertion into 'TopActors':

In [ ]:
%%sql
INSERT INTO TopActors (ActorID, FirstName, LastName, TotalFilms)
SELECT
    a.actor_id,
    a.first_name,
    a.last_name,
    COUNT(fa.film_id) AS TotalFilms
FROM
    actor a
JOIN
    film_actor fa ON a.actor_id = fa.actor_id
GROUP BY
    a.actor_id, a.first_name, a.last_name
HAVING
    COUNT(fa.film_id) > 10; -- Changed from > 5 to > 10

---

## 03 SELECT INTO

💻 **Task:** Modify the SELECT INTO statement to include not only the 'customer_id' and 'rental_count', but also the last rental date for each customer in July 2005. This will require adding a MAX function on the 'rental_date' column in the SELECT clause.

> **SQLite note:** As in the examples notebook, `SELECT ... INTO ActiveCustomersJuly2005` becomes `CREATE TABLE ActiveCustomersJuly2005 AS SELECT ...`.

Ensure the ActiveCustomersJuly2005 table doesn't already exist, then create a table to identify active customers in July 2005.

In [ ]:
%%sql
DROP TABLE IF EXISTS ActiveCustomersJuly2005;

CREATE TABLE ActiveCustomersJuly2005 AS
SELECT
    customer_id,
    COUNT(rental_id) AS rental_count,
    MAX(rental_date) AS last_rental_date
FROM
    rental
WHERE
    rental_date BETWEEN '2005-07-01' AND '2005-07-31'
GROUP BY
    customer_id
HAVING
    COUNT(rental_id) > 0;

Check the structure and some data from the newly created ActiveCustomersJuly2005 table.

In [ ]:
%%sql
SELECT *
FROM ActiveCustomersJuly2005
LIMIT 5;

---

## 04 UPDATE

💻 **Task:** Modify the UPDATE statement to also change the 'store_id' for the staff member with staff_id = 2 to 1, reflecting their move to a different store. This will involve also updating the 'store_id' column in the SET clause to a fixed value of 1.

Update the first name and store_id of the staff member with staff_id = 2.

In [ ]:
%%sql
UPDATE staff
SET
    first_name = 'John', -- Change first name to 'John'
    store_id = 1         -- Update store_id to 1
WHERE staff_id = 2;

Confirm the update by checking the updated information.

In [ ]:
%%sql
SELECT staff_id,
       first_name,
       store_id
FROM staff
WHERE staff_id = 2;

---

## 05 UPDATE with table joins

💻 **Task:** Modify the UPDATE statement to change the rental duration for classic films and set the rental rate to a flat rate of 1.99. This requires updating the 'rental_rate' column in the SET clause to 1.99, along with changing the rental duration.

> **SQLite note:** As in the examples notebook, `film` cannot be listed again in the `FROM` clause, so it is joined to `film_category` in the `WHERE` clause.

In [ ]:
%%sql
UPDATE film
SET
    rental_duration = 7, -- Change rental duration to 7 days
    rental_rate = 1.99   -- Set rental rate to a flat rate of $1.99
FROM film_category fc
INNER JOIN category c ON fc.category_id = c.category_id
WHERE film.film_id = fc.film_id
  AND c.name = 'Classics';

Confirm the update by rechecking the rental durations for classic films. Again, joining with 'film_category' and 'category' to filter by category name.

In [ ]:
%%sql
SELECT
    f.film_id,
    f.title,
    f.rental_duration,
    f.rental_rate
FROM film f
INNER JOIN film_category fc ON f.film_id = fc.film_id
INNER JOIN category c ON fc.category_id = c.category_id
WHERE c.name = 'Classics';

---

## 06 MERGE

💻 **Task:**

1. In Step 4 add a 'DELETE FROM... WHERE' on the temporary rental table to also simulate the deletion of rental records. Delete rental_id's 11 and 12.
2. Then after running the MERGE statement, inspect the output returned to confirm those two records were deleted.

*Set-up from the examples notebook:* Steps 1 to 3 create the temporary copy of the rental table and the staging table.

In [ ]:
%%sql
DROP TABLE IF EXISTS temp.temp_rental;

CREATE TEMP TABLE temp_rental (
    rental_id INTEGER PRIMARY KEY AUTOINCREMENT,
    rental_date DATETIME,
    inventory_id INT,
    customer_id INT,
    return_date DATETIME,
    staff_id INT,
    last_update DATETIME
);

INSERT INTO temp.temp_rental
SELECT * FROM rental;

In [ ]:
%%sql
DROP TABLE IF EXISTS STG_rental;

CREATE TABLE STG_rental (
    rental_id INT,
    rental_date DATETIME,
    inventory_id INT,
    customer_id INT,
    return_date DATETIME,
    staff_id INT,
    last_update DATETIME
);

INSERT INTO STG_rental
SELECT * FROM rental;

*Set-up from the examples notebook:* Step 4, the inserts and updates.

In [ ]:
%%sql
INSERT INTO temp.temp_rental (rental_date, inventory_id, customer_id, return_date, staff_id, last_update)
VALUES
(datetime('now'), 1, 10, NULL, 1, datetime('now')),
(datetime('now'), 2, 25, NULL, 1, datetime('now'));

In [ ]:
%%sql
UPDATE temp.temp_rental
SET return_date = datetime(rental_date, '+1 day'),
    last_update = datetime('now')
WHERE rental_id IN (1, 2);

**Answer, part 1:** delete specific records from the temporary rental table to simulate deletions.

In [ ]:
%%sql
DELETE FROM temp.temp_rental
WHERE rental_id IN (11, 12);

**Answer, part 2:** run the MERGE (in SQLite, its three statements) and inspect the output. The last statement, for `WHEN NOT MATCHED BY SOURCE`, returns the two deleted records, 11 and 12.

In [ ]:
%%sql
UPDATE STG_rental AS target
SET rental_date = source.rental_date,
    inventory_id = source.inventory_id,
    customer_id = source.customer_id,
    return_date = source.return_date,
    staff_id = source.staff_id,
    last_update = source.last_update
FROM temp.temp_rental AS source
WHERE target.rental_id = source.rental_id
  AND source.last_update > target.last_update
RETURNING 'UPDATE' AS Action, rental_id AS NewRentalID, return_date AS NewReturnDate, last_update AS NewLastUpdate;

In [ ]:
%%sql
INSERT INTO STG_rental (rental_id, rental_date, inventory_id, customer_id, return_date, staff_id, last_update)
SELECT source.rental_id, source.rental_date, source.inventory_id, source.customer_id, source.return_date, source.staff_id, source.last_update
FROM temp.temp_rental AS source
WHERE NOT EXISTS (
    SELECT 1 FROM STG_rental AS target WHERE target.rental_id = source.rental_id
)
RETURNING 'INSERT' AS Action, rental_id AS NewRentalID, rental_date AS NewRentalDate, customer_id AS NewCustomerID;

In [ ]:
%%sql
DELETE FROM STG_rental
WHERE NOT EXISTS (
    SELECT 1 FROM temp.temp_rental AS source WHERE source.rental_id = STG_rental.rental_id
)
RETURNING 'DELETE' AS Action, rental_id AS OldRentalID, rental_date AS OldRentalDate, customer_id AS OldCustomerID;

---

## 07 Subqueries

💻 **Task:** Modify the subquery to only include films with 30 or more rentals.

Step 1: Calculate the total number of rentals for each film, only including films with 30 or more rentals.

In [ ]:
%%sql
SELECT
    i.film_id,
    COUNT(r.rental_id) AS rental_count
FROM
    rental r
    INNER JOIN inventory i ON r.inventory_id = i.inventory_id
GROUP BY
    i.film_id
HAVING
    COUNT(r.rental_id) >= 30;

Step 2: List films with 30 or more rentals in each category, ordered by rental count.

In [ ]:
%%sql
SELECT
    c.name AS category_name,
    f.title AS film_title,
    r.rental_count
FROM
    (SELECT
        i.film_id,
        COUNT(r.rental_id) AS rental_count
    FROM
        rental r
        INNER JOIN inventory i ON r.inventory_id = i.inventory_id
    GROUP BY
        i.film_id
    HAVING
        COUNT(r.rental_id) >= 30) r
    INNER JOIN film_category fc ON r.film_id = fc.film_id
    INNER JOIN category c ON fc.category_id = c.category_id
    INNER JOIN film f ON r.film_id = f.film_id
ORDER BY
    c.name,
    r.rental_count DESC,
    f.title;

---

## 11 UNION ALL

💻 **Task:**

1. Replace UNION with UNION ALL in the final SELECT statement. Observe the difference in the result set.
2. Consider a scenario where a person can be both a customer and a staff member. Would using UNION vs UNION ALL affect the result set in this case?

In [ ]:
%%sql
SELECT
    first_name || ' ' || last_name AS name,
    'Customer' AS person_type
FROM
    customer
UNION ALL
SELECT
    first_name || ' ' || last_name AS name,
    'Staff' AS person_type
FROM
    staff
ORDER BY
    person_type DESC,
    name;

Changing to UNION ALL would not affect the result where a person is both a customer and a staff member, as the person_type would be different for both rows (i.e. the full row is not a duplicate).

However, if there are duplicate staff or customer names, these duplicate rows would be removed with UNION and kept with UNION ALL.

---

## 12 INTERSECT AND EXCEPT

💻 **Task:** Alter the INTERSECT query to also include the names of the actors, not just their IDs. This will require you to join with the actor table.

In [ ]:
%%sql
SELECT a.actor_id, a.first_name, a.last_name
FROM film_actor fa
INNER JOIN film_category fc ON fa.film_id = fc.film_id
INNER JOIN category c ON fc.category_id = c.category_id
INNER JOIN actor a ON fa.actor_id = a.actor_id
WHERE c.name = 'Action'
INTERSECT
SELECT a.actor_id, a.first_name, a.last_name
FROM film_actor fa
INNER JOIN film_category fc ON fa.film_id = fc.film_id
INNER JOIN category c ON fc.category_id = c.category_id
INNER JOIN actor a ON fa.actor_id = a.actor_id
WHERE c.name = 'Comedy';

---

## 13 Ranking Functions

💻 **Tasks:**

1. Add a new column in the CTE that counts the number of films in each category. Then, include this count in the final SELECT statement.
   📗 COUNT aggregate function: https://learn.microsoft.com/en-us/sql/t-sql/functions/count-transact-sql
2. Modify the ORDER BY clause in the window function to first rank categories by the number of films (added in the previous step), and then by TotalRevenue.
3. Change the RANK() function to DENSE_RANK() and observe how the rankings change if there were ties in the TotalRevenue. While there are no ties, would it make sense to use DENSE_RANK() in case ties occurred in later data updates?
   Explanation of DENSE_RANK(): https://learn.microsoft.com/en-us/sql/t-sql/functions/dense-rank-transact-sql

`COUNT(DISTINCT f.film_id)` counts each film once. The CTE joins every rental and payment, so a plain `COUNT(f.film_id)` would count payments, not films.

In [ ]:
%%sql
WITH CategoryRevenue AS (
    SELECT
        c.name AS Category,
        ROUND(SUM(p.amount), 2) AS TotalRevenue,
        COUNT(DISTINCT f.film_id) AS FilmCount
    FROM
        rental r
        JOIN inventory i ON r.inventory_id = i.inventory_id
        JOIN film f ON i.film_id = f.film_id
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
        JOIN payment p ON r.rental_id = p.rental_id
    GROUP BY
        c.name
)
SELECT
    Category,
    TotalRevenue,
    FilmCount,
    DENSE_RANK() OVER (ORDER BY FilmCount DESC, TotalRevenue DESC) AS Rank
FROM
    CategoryRevenue
ORDER BY
    Rank;

---

## 14 LAG FUNCTION

💻 **Tasks:**

1. Update the ORDER BY clause in the LAG function to explore how different sorting orders (e.g., DESC instead of ASC) impact the PreviousMonthTotal calculation.
2. Extend the CTE to include a count of the total number of transactions for each month, then include this count in your final SELECT statement. This will provide additional context to the revenue figures.

> **SQLite note:** As in the examples notebook, `FORMAT(EOMONTH(payment_date), 'yyyy-MM')` becomes `strftime('%Y-%m', payment_date)`.

In [ ]:
%%sql
WITH MonthlyTotals AS (
    SELECT
        strftime('%Y-%m', payment_date) AS Month,
        ROUND(SUM(amount), 2) AS TotalAmount,
        COUNT(*) AS TransactionCount
    FROM
        payment
    GROUP BY
        strftime('%Y-%m', payment_date)
)
SELECT
    Month,
    TotalAmount,
    LAG(TotalAmount, 1, 0) OVER (ORDER BY Month DESC) AS PreviousMonthTotal,
    TransactionCount,
    LAG(TransactionCount, 1, 0) OVER (ORDER BY Month DESC) AS PreviousMonthTransactionCount
FROM
    MonthlyTotals
ORDER BY
    Month;

---

## 15 STORED PROCEDURES

💻 **Tasks:**

1. Update the stored procedure to return the average rental revenue per film along with the existing columns. This will provide more depth to the analysis.
2. Change the ORDER BY clause to first sort by the new average rental revenue column in descending order, then by TotalRentals.
3. Modify the stored procedure to add a parameter that also allows filtering films by a minimum rental count threshold. This will add flexibility to the report generation.

The original T-SQL answer was:

```sql
CREATE PROCEDURE sp_TopPerformingFilms
    @TopN INT = 10,                       -- Parameter to specify the number of top films to return
    @MinRentalCount INT = 0               -- Parameter to filter films by a minimum rental count threshold
AS
BEGIN
    SET NOCOUNT ON;

    SELECT TOP (@TopN)
        f.film_id,
        f.title,
        COUNT(r.rental_id) AS TotalRentals,
        SUM(p.amount) AS TotalRevenue,
        AVG(p.amount) AS AvgRevenuePerRental
    FROM
        rental r
        INNER JOIN inventory i ON r.inventory_id = i.inventory_id
        INNER JOIN film f ON i.film_id = f.film_id
        INNER JOIN payment p ON r.rental_id = p.rental_id
    GROUP BY
        f.film_id, f.title
    HAVING
        COUNT(r.rental_id) >= @MinRentalCount    -- Filter films by the minimum rental count threshold
    ORDER BY
        AvgRevenuePerRental DESC,                -- Sort by average revenue per film first
        TotalRentals DESC;                       -- Then by total rentals
END;
GO

EXEC sp_TopPerformingFilms
```

> **SQLite note:** As in the examples notebook, the stored procedure is a Python function, and its parameters are passed to the query as `:TopN` and `:MinRentalCount`.

In [ ]:
import pandas as pd

def sp_TopPerformingFilms(TopN=10,           # Parameter to specify the number of top films to return
                          MinRentalCount=0):  # Parameter to filter films by a minimum rental count threshold
    # Query to find top N performing films based on rental count, total revenue, and average revenue per film
    query = """
    SELECT
        f.film_id,
        f.title,
        COUNT(r.rental_id) AS TotalRentals,
        ROUND(SUM(p.amount), 2) AS TotalRevenue,
        ROUND(AVG(p.amount), 2) AS AvgRevenuePerRental
    FROM
        rental r
        INNER JOIN inventory i ON r.inventory_id = i.inventory_id
        INNER JOIN film f ON i.film_id = f.film_id
        INNER JOIN payment p ON r.rental_id = p.rental_id
    GROUP BY
        f.film_id, f.title
    HAVING
        COUNT(r.rental_id) >= :MinRentalCount    -- Filter films by the minimum rental count threshold
    ORDER BY
        AvgRevenuePerRental DESC,                -- Sort by average revenue per film first
        TotalRentals DESC                        -- Then by total rentals
    LIMIT :TopN;
    """
    return pd.read_sql_query(query, con, params={'TopN': TopN, 'MinRentalCount': MinRentalCount})

Execute the updated "stored procedure" to retrieve the report.

In [ ]:
sp_TopPerformingFilms()

---

## 16 Functions

💻 **Task:** Update the function everywhere it is called so that the late fee per day is now 2.50.

*Set-up from the examples notebook:* create the late fee function (in SQLite, a Python function registered with the connection).

In [ ]:
from datetime import datetime, timedelta

def fn_CalculateLateFee(RentalDate, ReturnDate, RentalDuration, RatePerDay):
    # A rental that has not been returned has no return date (NULL), so no late fee yet.
    if ReturnDate is None:
        return 0
    due_date = datetime.fromisoformat(RentalDate) + timedelta(days=RentalDuration)
    # Like T-SQL's DATEDIFF(DAY, ...), count the calendar days between the two dates
    DaysOverdue = (datetime.fromisoformat(ReturnDate).date() - due_date.date()).days

    if DaysOverdue > 0:
        return round(DaysOverdue * RatePerDay, 2)

    return 0

con.create_function('fn_CalculateLateFee', 4, fn_CalculateLateFee, deterministic=True)

**Answer:**

In [ ]:
%%sql
SELECT
    r.rental_id,
    c.customer_id,
    c.first_name || ' ' || c.last_name AS CustomerName,
    r.rental_date,
    r.return_date,
    f.rental_duration,
    fn_CalculateLateFee(r.rental_date, r.return_date, f.rental_duration, 2.50) AS LateFee
FROM
    rental r
    INNER JOIN customer c ON r.customer_id = c.customer_id
    INNER JOIN inventory i ON r.inventory_id = i.inventory_id
    INNER JOIN film f ON i.film_id = f.film_id
WHERE
    fn_CalculateLateFee(r.rental_date, r.return_date, f.rental_duration, 2.50) > 0
ORDER BY
    LateFee DESC;

*End of answers*